### 08. Building a Collaborative Filtering Recommendation Model

**Objective:** Build a **User-Based Collaborative Filtering** recommender that suggests TED Talks based on similar users' rating patterns.

**Input:**
- `../data/interaction/user_interactions.csv` (from Task 7: 500 users x 200 talks, ~9,800 ratings)
- `../data/processed/ted_talks_cleaned.csv` (for talk titles)

**⚠ Simulated Data:** The interaction data was simulated in Task 7 for learning purposes. No real user data is used.

**Approach:**
1. Load interactions and build the user-item matrix.
2. Compute each user's **mean rating** over the talks they actually rated.
3. **Mean-center** the matrix (`rating - user_mean`), then fill missing values with **0**.
   A 0 now means "this user's average", not "rated 0".
4. Compute **cosine similarity** on the mean-centered vectors.
5. For a target user, find **top-K similar users** (K=20).
6. Predict a deviation for unrated talks from neighbours, add the target user's mean back,
   and **clip to [1, 5]**.
7. Return the **top-N** recommendations, breaking score ties by `n_raters`.

In [69]:
import numpy as np
import pandas as pd
from pathlib import Path

pd.set_option("display.max_colwidth", 100)
pd.set_option("display.width", 160)

##### Load the Interaction Dataset

In [70]:
interactions = pd.read_csv("../data/interaction/user_interactions.csv")
talks = pd.read_csv("../data/processed/ted_talks_cleaned.csv")[["talk_id", "title"]]

print(f"Interactions: {interactions.shape[0]} rows")
print(f"Unique users: {interactions['user_id'].nunique()}")
print(f"Unique talks: {interactions['talk_id'].nunique()}")
print(f"Rating range: {interactions['rating'].min()} – {interactions['rating'].max()}")
print(f"Ratings per user: min={interactions.groupby('user_id').size().min()}, "
      f"median={interactions.groupby('user_id').size().median():.0f}")
interactions.head()

Interactions: 9778 rows
Unique users: 500
Unique talks: 200
Rating range: 1 – 5
Ratings per user: min=7, median=19


,user_id,talk_id,rating,liked,saved,watch_pct,timestamp
0,U0001,27155,3,False,False,0.99,2023-07-27
1,U0001,1375,3,False,False,0.70,2023-01-20
2,U0001,13029,4,True,False,1.00,2023-03-09
3,U0001,19266,2,False,False,0.42,2024-04-05
4,U0001,2722,2,False,False,0.43,2024-03-27


##### Build the User-Item Interaction Matrix
- **Rows:** users
- **Columns:** talks
- **Values:** rating (1–5)
- **NaN** = user has not rated the talk

In [71]:
user_item_matrix = interactions.pivot_table(
    index="user_id",
    columns="talk_id",
    values="rating",
    aggfunc="mean",
)

print(f"Matrix shape: {user_item_matrix.shape}  (users × talks)")
print(f"Filled entries:  {user_item_matrix.notna().sum().sum()}")
print(f"Missing entries: {user_item_matrix.isna().sum().sum()}")
print(f"Sparsity:        {user_item_matrix.isna().mean().mean():.2%}")

user_item_matrix.iloc[:8, :8]

Matrix shape: (500, 200)  (users × talks)
Filled entries:  9778
Missing entries: 90222
Sparsity:        90.22%


talk_id,27,44,46,71,79,91,131,190
user_id,,,,,,,,
U0001,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
U0002,2.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
U0003,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
U0004,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
U0005,NaN,NaN,NaN,3.0,NaN,NaN,NaN,NaN
U0006,NaN,NaN,NaN,NaN,NaN,3.0,NaN,3.0
U0007,NaN,NaN,NaN,4.0,NaN,NaN,NaN,NaN
U0008,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


##### Calculate User Similarity (Cosine, Mean-Centered)

Raw ratings are on a 1–5 scale, so two users who both rate everything around 4 look similar even if
they disagree on *relative* preferences. **Mean-centering** removes each user's personal baseline:
`centered = rating - user_mean`. Missing entries become 0, which now means "average for this user".

Cosine similarity on these centered vectors captures whether users agree on *which talks are above or
below their own average* - a much better signal than raw co-rating.

In [72]:
from sklearn.metrics.pairwise import cosine_similarity

# Per-user mean of the ratings they actually gave (NaN-aware)
user_means = user_item_matrix.mean(axis=1)

# Mean-center each user's ratings, then fill unrated with 0
# (0 now means "average for this user", not "rated 0")
centered_matrix = user_item_matrix.sub(user_means, axis=0).fillna(0.0)

# Cosine similarity on the mean-centered vectors
user_sim = cosine_similarity(centered_matrix.values, dense_output=True)
np.fill_diagonal(user_sim, 0.0)  # exclude self

user_sim_df = pd.DataFrame(
    user_sim,
    index=user_item_matrix.index,
    columns=user_item_matrix.index,
)

print(f"User similarity matrix shape: {user_sim_df.shape}")
print(f"Similarity range: {user_sim_df.values.min():.4f} – {user_sim_df.values.max():.4f}")
print(f"Average pairwise similarity: {user_sim_df.values[np.triu_indices_from(user_sim_df.values, k=1)].mean():.4f}")

# Preview the first 6x6 block
user_sim_df.iloc[:6, :6].round(3)

User similarity matrix shape: (500, 500)
Similarity range: -0.5958 – 0.4730
Average pairwise similarity: 0.0006


user_id,U0001,U0002,U0003,U0004,U0005,U0006
user_id,,,,,,
U0001,0.000,-0.043,0.000,0.020,0.054,-0.058
U0002,-0.043,0.000,-0.047,-0.159,0.001,-0.067
U0003,0.000,-0.047,0.000,-0.035,0.000,0.149
U0004,0.020,-0.159,-0.035,0.000,0.000,-0.004
U0005,0.054,0.001,0.000,0.000,0.000,-0.013
U0006,-0.058,-0.067,0.149,-0.004,-0.013,0.000


> **Note on similarity scores:** After mean-centering, cosine similarity is **signed**: average pairwise similarity is **0.0006** (median 0.000), with a range of **−0.596 to 0.473**. About **41.9%** of user pairs are positive and **41.5%** are negative - a near-zero mean is expected because every centered user vector sums to zero, so random pairs have no systematic alignment. Positive values mean two users agree on which talks sit above/below their own averages; negative values mean they disagree. **84.8% of user pairs share at least one co-rated talk** (up to 13), so every user has a usable neighbourhood.

##### Identify Similar Users for a Sample User
Let's look at User `U0001`'s top-10 most similar users.

In [73]:
SAMPLE_USER = "U0001"

sims = user_sim_df.loc[SAMPLE_USER].sort_values(ascending=False)
top_similar = sims.head(10)

print(f"Top 10 users most similar to {SAMPLE_USER}:")
print(top_similar.round(4))

Top 10 users most similar to U0001:
user_id
U0281    0.2849
U0087    0.2712
U0216    0.2536
U0037    0.2345
U0360    0.2221
U0174    0.2218
U0078    0.2089
U0211    0.1803
U0454    0.1772
U0092    0.1694
Name: U0001, dtype: float64


##### Recommendation Function

**`recommend_for_user(user_id, top_n=5, k_neighbors=20)`**

Steps:
1. Validate the user exists.
2. Take the top-K most similar users (neighbours).
3. For each talk the target user **has not rated**, compute a predicted **deviation**:
   $$\widehat{dev}_{u,i} = \frac{\sum_{v \in N} \text{sim}(u,v)\,(r_{v,i} - \bar{r}_v)}{\sum_{v \in N} |\text{sim}(u,v)|}$$
   where N = neighbours who actually rated talk *i*.
4. Add the target user's mean back and clip to the rating range:
   $$\hat{r}_{u,i} = \text{clip}\left(\bar{r}_u + \widehat{dev}_{u,i},\ 1,\ 5\right)$$
5. Sort by predicted score, breaking ties by `n_raters` (how many neighbours rated the talk), take top-N.
6. Attach talk titles.

In [74]:
def recommend_for_user(user_id, top_n=5, k_neighbors=20):
    """
    Mean-centered user-based collaborative filtering recommender.

    Parameters:
    - user_id (str): The user to recommend for.
    - top_n (int): Number of recommendations (default 5).
    - k_neighbors (int): Number of similar users to consider (default 20).

    Returns:
    - pd.DataFrame: columns [Rank, talk_id, title, Predicted Score]
    - None: if user is invalid or has no interactions.
    """
    # Validate user
    if user_id not in user_item_matrix.index:
        print(f"Invalid user_id: '{user_id}'. Not found in the interaction dataset.")
        return None

    # Validate top_n
    if not isinstance(top_n, int) or top_n < 1 or top_n > 20:
        print("Error: top_n must be an integer between 1 and 20.")
        return None

    # Talks the target user has already rated
    user_ratings = user_item_matrix.loc[user_id]
    already_rated = user_ratings.dropna().index
    if len(already_rated) == 0:
        print(f"User '{user_id}' has no interaction history - cold-start. Cannot compute CF recommendations.")
        return None

    # Similarity scores to all other users
    sims = user_sim_df.loc[user_id].drop(user_id)

    # Top-K neighbours
    neighbours = sims.sort_values(ascending=False).head(k_neighbors)
    neighbour_ids = neighbours.index
    neighbour_weights = neighbours.values

    # Mean-centered ratings of the neighbours (0 where unrated)
    neighbour_centered = centered_matrix.loc[neighbour_ids]
    candidate_mask = user_ratings.isna()
    neighbours_with_data = user_item_matrix.loc[neighbour_ids].notna()
    target_mean = float(user_means[user_id])

    # Mean-centered deviation predictions
    pred_data = []
    for talk_id in user_item_matrix.columns:
        if not candidate_mask[talk_id]:
            continue  # user already rated it
        rated_by = neighbours_with_data[talk_id]
        if not rated_by.any():
            continue

        weights = neighbour_weights[rated_by.values]
        centered_ratings = neighbour_centered.loc[rated_by, talk_id].values

        denom = np.sum(np.abs(weights))
        if denom <= 0:
            continue

        pred_dev = np.dot(weights, centered_ratings) / denom
        pred_rating = float(np.clip(target_mean + pred_dev, 1.0, 5.0))
        pred_data.append((talk_id, pred_rating, int(rated_by.sum())))

    if not pred_data:
        print(f"No recommendations could be generated for '{user_id}'.")
        return None

    # Sort by predicted rating, breaking ties by number of neighbours who rated
    pred_df = pd.DataFrame(pred_data, columns=["talk_id", "Predicted Score", "n_raters"])
    pred_df = pred_df.sort_values(["Predicted Score", "n_raters"], ascending=[False, False]).head(top_n)
    pred_df.insert(0, "Rank", range(1, len(pred_df) + 1))
    pred_df["Predicted Score"] = pred_df["Predicted Score"].round(4)

    # Attach titles
    rec_df = pred_df[["Rank", "talk_id", "Predicted Score"]]
    rec_df = rec_df.merge(talks, on="talk_id", how="left")
    rec_df = rec_df[["Rank", "talk_id", "title", "Predicted Score"]]
    return rec_df

##### Test the Recommendation Function

Test with three different users and one invalid user.

In [75]:
# Test 1 - U0001
recommend_for_user("U0001", top_n=5)

,Rank,talk_id,title,Predicted Score
0,1,912,global population growth box by box,3.6303
1,2,10975,you don t have to be an expert to solve big problems,3.3198
2,3,1534,a test for parkinson s with a phone call,3.3083
3,4,24347,can you solve the control room riddle,3.3083
4,5,201,the lost art of letter writing,3.3037


>**Note**: Mean-centering replaces the old 4.0/5.0 ties with graded predictions. U0001's top recommendation scores **3.63**; the next three sit at **3.31–3.32** and the fifth at **3.30**. Where scores are exactly tied, `n_raters` (how many of the 20 neighbours rated the talk) breaks the tie deterministically. All scores are clipped to **[1, 5]**.

In [76]:
# Test 2 - U0002 (29 ratings, among the most active users in Task 7)
recommend_for_user("U0002", top_n=5)

,Rank,talk_id,title,Predicted Score
0,1,91,invest in africa s own solutions,4.7186
1,2,32250,will there ever be a mile high skyscraper,4.2102
2,3,1773,the dangers of willful blindness,4.0365
3,4,2707,a burial practice that nourishes the planet,4.0086
4,5,1077,the future of business is the mesh,3.9939


In [77]:
# Test 3 - U0003
recommend_for_user("U0003", top_n=5)

,Rank,talk_id,title,Predicted Score
0,1,42461,new colossus,4.6234
1,2,55793,could a breathalyzer detect cancer,3.8643
2,3,1656,a new kind of job market,3.8262
3,4,2508,the beauty of being a misfit,3.7964
4,5,27155,how to motivate people to do good for others,3.7249


In [78]:
# Test 4 - Invalid user
recommend_for_user("U9999", top_n=5)

Invalid user_id: 'U9999'. Not found in the interaction dataset.


##### Verification Checks

We verify the function meets the four required checks:
1. Generates recommendations for valid users.
2. Excludes talks already rated by the user.
3. Returns sorted results.
4. Different users get different recommendations.

In [79]:
def verify_recommendations(user_id, top_n=5):
    print(f"\nVerifying recommendations for {user_id}...")
    recs = recommend_for_user(user_id, top_n=top_n)
    if recs is None:
        return

    # 1. Excludes already-rated talks
    rated = set(user_item_matrix.loc[user_id].dropna().index)
    overlap = set(recs["talk_id"]) & rated
    print(f"  ✔ Talks already rated by user: {len(overlap)} (expected 0)")

    # 2. Sorted by predicted score
    scores = recs["Predicted Score"].values
    is_sorted = all(scores[i] >= scores[i + 1] for i in range(len(scores) - 1))
    print(f"  ✔ Sorted by predicted score: {is_sorted}")

    # 3. Returns exactly top_n rows
    print(f"  ✔ Returned rows: {len(recs)} (requested {top_n})")

for uid in ["U0001", "U0002", "U0003"]:
    verify_recommendations(uid, top_n=5)


Verifying recommendations for U0001...
  ✔ Talks already rated by user: 0 (expected 0)
  ✔ Sorted by predicted score: True
  ✔ Returned rows: 5 (requested 5)

Verifying recommendations for U0002...
  ✔ Talks already rated by user: 0 (expected 0)
  ✔ Sorted by predicted score: True
  ✔ Returned rows: 5 (requested 5)

Verifying recommendations for U0003...
  ✔ Talks already rated by user: 0 (expected 0)
  ✔ Sorted by predicted score: True
  ✔ Returned rows: 5 (requested 5)


In [80]:
# 4. Different users -> different recommendations
rec_a = recommend_for_user("U0001", top_n=5)
rec_b = recommend_for_user("U0002", top_n=5)

if rec_a is None or rec_b is None:
    print("Cannot compare: at least one user has no CF predictions.")
else:
    overlap_ids = set(rec_a["talk_id"]) & set(rec_b["talk_id"])
    print(f"Overlap between U0001 and U0002 recommendations: {len(overlap_ids)} talks")
    print(f"U0001 got: {rec_a['talk_id'].tolist()}")
    print(f"U0002 got: {rec_b['talk_id'].tolist()}")

Overlap between U0001 and U0002 recommendations: 0 talks
U0001 got: [912, 10975, 1534, 24347, 201]
U0002 got: [91, 32250, 1773, 2707, 1077]


##### Answers to Required Questions

**1. What is Collaborative Filtering?**
A recommendation technique that predicts a user's preferences based on the behaviour of *other similar users* - without analysing the content of the items. "Users like you liked X, so you might too."

**2. How is it different from Content-Based Filtering?**
| Aspect | Content-Based | Collaborative |
|---|---|---|
| Uses | Item features (title, tags, transcript) | User-item interactions (ratings) |
| Asks | "What is this talk about?" | "Who else liked this talk?" |
| Personalisation | Per-user preferences on item features | Per-user preferences on other users' behaviour |
| New item | Works well (features available) | Fails (no ratings yet - cold item) |
| New user | Works (needs a seed item) | Fails (no history - cold user) |

**3. What is a User-Item Interaction Matrix?**
A 2-D table where **rows = users, columns = talks, values = ratings**. A missing value means "user did not rate that talk."

**4. Why do we calculate similarity between users?**
To find users whose rating patterns match the target user. Their ratings are then used to predict scores for talks the target user hasn't seen.

**5. What does a cosine similarity score represent?**
The **angle** between two users' mean-centered rating vectors. Because each user's average is subtracted first, the vector captures *relative* preference (above/below their own baseline), and missing entries become 0 = "average". Higher scores mean the two users agree more on which talks are better than their personal norm.

**6. Why should we exclude talks the user already rated?**
Recommending something they've already seen is useless. CF should only surface *new* items.

**7. What is a predicted rating?**
An estimate of how much the target user would rate a given talk: the similarity-weighted average of neighbours' *deviations from their own means*, added back to the target user's mean and clipped to [1, 5].

**8. What happens when a new user has no interaction history?**
The function returns an error / `None`. There's no data to compute a mean, similarity, or predictions - the standard **cold-start** failure mode.

**9. What is the Cold-Start Problem?**
The inability to make good recommendations when:
- **New user:** no rating history -> no similarity signal.
- **New item:** no ratings -> never recommended.
Typical mitigations: ask for onboarding preferences, fall back to popularity, or use content-based features (hybrid).

**10. What are the limitations of User-Based CF?**
- **Cold start** for new users and new items.
- **Sparsity:** most user-user pairs share very few rated items -> unstable similarity.
- **Scalability:** O(users²) similarity computation - expensive at millions of users.
- **Popularity bias:** popular talks dominate; niche tastes are under-served.
- **Shilling / manipulation:** fake ratings can manipulate neighbours.
- **User tastes change over time** - similarity based on old ratings may go stale.
- **Mean-centering helps but is not a cure:** it removes per-user scale bias, yet pairs with only one or two co-rated talks still get noisy similarity.

##### Observations

1. **Similar users are identified** by cosine similarity on **mean-centered** rating vectors. With 500 users and ~20 ratings each, 84.8% of user pairs share at least one co-rated talk (up to 13), so every user has a usable neighbourhood.

2. **Recommendations are generated** as a similarity-weighted average of neighbours' *deviations from their own means*, added back to the target user's mean and clipped to [1, 5]. All 500 users receive a full top-5 list.

3. **Different users receive different recommendations.** U0001, U0002, and U0003 have different neighbour sets and different rated talks; U0001 and U0002 share **0 of 5** recommendations.

4. **Mean-centering removes per-user scale bias.** Raw ratings run from user means of **2.10 to 3.23** (median 2.67). A generous rater who gives mostly 4s and a harsh rater who gives mostly 2s can still have identical *relative* preferences; centering lets their similarity reflect that agreement instead of their baseline.

5. **Limitations observed:**
   - **Cold-start:** a brand-new user with no ratings cannot receive CF recommendations (handled with an error + fallback message). All 500 simulated users have ratings, so none hit this case here.
   - **Sparsity:** 90.2% of the matrix is missing. Similarity is therefore noisy, and **41.5% of pairs are actually negative** - the neighbourhood is informative but not uniformly positive.
   - **Thin overlap:** mean-centering cannot rescue a pair that shares only one or two talks; a single disagreement can flip the sign of the similarity.
   - **Popularity bias:** talks rated by many users are more likely to appear in predictions simply because they have data.
   - **No item features used:** two very different talks with similar rating patterns look identical to the model.

6. **Next step (Task 9):** Combine this CF model with the content-based recommender from Task 6 into a **hybrid** system - CF handles personalisation, content handles cold-start and diverse item descriptions.

##### Save Collaborative Filtering Model
Persist the user-item matrix, per-user means, similarity matrix, and neighbour count in a single
`cf_model.joblib` bundle for reuse in Task 9 (Hybrid Recommendation) and Task 10 (Evaluation).

In [81]:
from pathlib import Path
import joblib

PROCESSED_DIR = Path("../data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

CF_MODEL_PATH = PROCESSED_DIR / "cf_model.joblib"
cf_model = {
    "user_item": user_item_matrix,
    "user_means": user_means,
    "sim_df": user_sim_df,
    "k": 20,
}
joblib.dump(cf_model, CF_MODEL_PATH)

print(f"Saved: {CF_MODEL_PATH}")
print(f"  user_item:  {user_item_matrix.shape}")
print(f"  user_means: {user_means.shape}")
print(f"  sim_df:     {user_sim_df.shape}")
print(f"  k:          {cf_model['k']}")

Saved: ..\data\processed\cf_model.joblib
  user_item:  (500, 200)
  user_means: (500,)
  sim_df:     (500, 500)
  k:          20


#### 📄 Task Report

The written report for this notebook lives at [`reports/task_08_observations.md`](../reports/task_08_observations.md).

- Markdown: [`reports/task_08_observations.md`](../reports/task_08_observations.md)
- PDF: [`reports/pdf/task_08_observations.pdf`](../reports/pdf/task_08_observations.pdf)

If a link does not open inside your notebook viewer, open the file directly from the project root.
